In [16]:
import pandas as pd
df_ground_truth=pd.read_csv("data/ground_truth-new.csv")

In [17]:
df_ground_truth.tail()

,question,document
390,I'm getting a version conflict where requests ...,4b30b918bc
391,What should I do if pip is installing an older...,4b30b918bc
392,How can I force install requests v2.32 from Gi...,4b30b918bc
393,I'm stuck with requests v2.28 and lancedb is c...,4b30b918bc
394,Why is my requests library installing the wron...,4b30b918bc


In [18]:
ground_truth = df_ground_truth.to_dict(orient="records")

In [19]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [20]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [32]:
q = ground_truth[0]
q
doc_id = q["document"]
results = text_search(query=q["question"])

In [31]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

e5d8a2c761 == 74eb249bbf: False
85384a18e5 == 74eb249bbf: False
d4f7c08ea1 == 74eb249bbf: False
74eb249bbf == 74eb249bbf: True
c2903069a0 == 74eb249bbf: False


In [33]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[0, 0, 0, 0, 0]

In [34]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [36]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)

Is it too late to enroll in the course right now?


[0, 0, 0, 0, 0]

In [43]:
q = ground_truth[17]
print(q["question"])
compute_relevance_text(q)

How do I know if the free GPU hours on a platform are reset daily, weekly, or monthly?


[0, 0, 1, 0, 0]

In [44]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [45]:
ground_truth_sample = ground_truth[:15]


In [46]:
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [48]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [49]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [50]:
relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [51]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/395 [00:00<?, ?it/s]